In [3]:
"""
build_tender_features.py
========================
Створює датасет на рівні ТЕНДЕРА з ознаками що відомі В МОМЕНТ ОГОЛОШЕННЯ
плюс мітку ДАСУ. Працює на всіх ~12 млн тендерах за 2022-2025.

Принципова відмінність:
    Жодних ознак що з'являються ПІСЛЯ публікації тендера —
    бо мета прогнозувати ризик ДО того як тендер закрився.

Залишаємо лише:
    - метод закупівлі, категорія, CPV, критерій відбору
    - сума тендера, дата публікації, день тижня, місяць
    - кількість лотів, документів, тривалість прийому пропозицій
    - чи замаскований замовник
    - buyer_violation_rate (рейтинг замовника з історії)

Викидаємо:
    - is_single_bidder, number_of_tenderers, number_of_bids
    - price_change_pct, near_zero_discount
    - award_concentration, award_value, discount_percentage_avg
    - has_cancelled_awards, has_unsuccessful_awards, has_multiple_awards
    - value_realization

Мітка:
    dasu_label = 1 якщо тендер мав моніторинг ДАСУ з порушенням
    dasu_label = 0 у всіх інших випадках

Це СЛАБКА мітка — серед 0 точно є тендери з фактичними порушеннями
яких ДАСУ просто не помітила. У курсовій це обмеження обговорюється.

Вихід:
    features_tenders_full.parquet
"""

import glob
import os
import warnings

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════
# НАЛАШТУВАННЯ
# ══════════════════════════════════════════════════════════════

CHUNKSIZE        = 300_000
HIGH_RISK_CPV    = {9, 45, 71, 72, 73, 79}

TENDERS_GLOB     = "../data/DataSet/tenders_*.csv"
DASU_LABELS_PATH = "dasu_labels.parquet"
OUTPUT_PATH      = "features_tenders_full.parquet"

NEEDED_COLS = [
    "tender_id", "buyer_id", "procuring_entity_id",
    "tender_start_date", "tender_end_date",
    "procurement_method", "main_procurement_category", "award_criteria",
    "main_cpv_2_digit",
    "tender_value",
    "number_of_items", "number_of_documents",
    "is_buyer_masked",
    "is_weekend", "is_q4", "is_december",
]


# ══════════════════════════════════════════════════════════════
# 1. Завантаження міток ДАСУ
# ══════════════════════════════════════════════════════════════

def load_dasu_labels() -> set:
    """Повертає множину tender_id з зафіксованим порушенням."""
    if not os.path.exists(DASU_LABELS_PATH):
        raise FileNotFoundError(
            f"Не знайдено {DASU_LABELS_PATH}. "
            "Спочатку запусти fetch_dasu_monitorings.py"
        )
    df = pd.read_parquet(DASU_LABELS_PATH, columns=["tender_id", "dasu_label"])
    df_pos = df[df["dasu_label"] == 1].copy()
    print(f"[ДАСУ] Завантажено {len(df):,} тендерів з моніторингом")
    print(f"[ДАСУ] З них з порушеннями: {len(df_pos):,}")
    return set(df_pos["tender_id"].astype(str).str.strip().tolist())


# ══════════════════════════════════════════════════════════════
# 2. Обробка одного чанка
# ══════════════════════════════════════════════════════════════

def process_chunk(chunk: pd.DataFrame, violation_ids: set) -> pd.DataFrame:
    """Перетворює сирий чанк у рядки готових ознак моменту публікації."""
    c = chunk.copy()

    # ── Очистка ідентифікаторів ──────────────────────────────
    c["tender_id"] = c["tender_id"].astype(str).str.strip()
    c["buyer_id"] = (
        c["buyer_id"].astype(str).str.strip().str.zfill(8)
        .replace({"nan": np.nan, "": np.nan})
    )
    c["procuring_entity_id"] = (
        c["procuring_entity_id"].astype(str).str.strip().str.zfill(8)
        .replace({"nan": np.nan, "": np.nan})
    )
    c["_key"] = c["buyer_id"].where(c["buyer_id"].notna(),
                                     c["procuring_entity_id"])

    # ── Дати ─────────────────────────────────────────────────
    c["tender_start_date"] = pd.to_datetime(
        c["tender_start_date"], errors="coerce", utc=True
    ).dt.tz_localize(None)

    c = c[
        c["_key"].notna() & c["_key"].ne("nan") & c["tender_start_date"].notna()
    ].copy()
    COMPETITIVE_METHODS = {"open", "selective"}
    c = c[c["procurement_method"].astype(str).str.lower().isin(COMPETITIVE_METHODS)]
    if c.empty:
        return pd.DataFrame()

    # ── Мітка ДАСУ ───────────────────────────────────────────
    c["dasu_label"] = c["tender_id"].isin(violation_ids).astype("int8")

    # ── Submission window ────────────────────────────────────
    if "tender_end_date" in c.columns:
        end_dt = pd.to_datetime(
            c["tender_end_date"], errors="coerce", utc=True
        ).dt.tz_localize(None)
        submission_window = (end_dt - c["tender_start_date"]).dt.days
    else:
        submission_window = pd.Series(np.nan, index=c.index)

    # ── Метод закупівлі ──────────────────────────────────────
    method_map = {"limited": 2, "selective": 1, "open": 0,
                  "reporting": 0, "negotiation": 2}
    proc_method_enc = (
        c["procurement_method"].astype(str).map(method_map)
        .fillna(0).astype("int8")
    )

    # ── Категорія, критерій ──────────────────────────────────
    cat = c["main_procurement_category"].astype(str)
    is_works    = (cat == "works").astype("int8")
    is_services = (cat == "services").astype("int8")
    non_price   = (c["award_criteria"].astype(str) != "lowestCost").astype("int8")

    # ── CPV ──────────────────────────────────────────────────
    cpv_num = pd.to_numeric(c["main_cpv_2_digit"], errors="coerce")
    is_high_risk_cpv = cpv_num.isin(HIGH_RISK_CPV).astype("int8")

    # ── Числові ────────────────────────────────────────────────
    tv  = pd.to_numeric(c["tender_value"],         errors="coerce")
    doc = pd.to_numeric(c["number_of_documents"],  errors="coerce")
    itm = pd.to_numeric(c["number_of_items"],      errors="coerce")
    log_tender_value = np.log1p(tv.fillna(0))

    # ── Збираємо ─────────────────────────────────────────────
    out = pd.DataFrame({
        "tender_id":              c["tender_id"].values,
        "buyer_id":               c["_key"].values,
        "tender_start_date":         c["tender_start_date"].values,
        "main_cpv_2_digit":       cpv_num.fillna(0).astype("int16").values,

        "dasu_label":             c["dasu_label"].values,

        "proc_method_enc":        proc_method_enc.values,
        "non_price_criteria":     non_price.values,
        "is_works":               is_works.values,
        "is_services":            is_services.values,
        "is_high_risk_cpv":       is_high_risk_cpv.values,
        "log_tender_value":       log_tender_value.astype("float32").values,
        "submission_window_days": submission_window.fillna(-1).astype("float32").values,
        "has_submission_window":  submission_window.notna().astype("int8").values,
        "number_of_items":        itm.fillna(0).astype("float32").values,
        "number_of_documents":    doc.fillna(0).astype("float32").values,
        "is_buyer_masked":        c["is_buyer_masked"].fillna(0).astype("int8").values,
        "is_weekend":             c["is_weekend"].fillna(0).astype("int8").values,
        "is_q4":                  c["is_q4"].fillna(0).astype("int8").values,
        "is_december":            c["is_december"].fillna(0).astype("int8").values,
    })
    return out


# ══════════════════════════════════════════════════════════════
# 3. Рейтинг замовника
# ══════════════════════════════════════════════════════════════

def add_buyer_violation_rate(df: pd.DataFrame) -> pd.DataFrame:
    """
    Додає колонку buyer_violation_rate — згладжена частка порушень
    замовника. Згладжування Лапласа: (порушень + 1) / (всього + 10).

    УВАГА: data leakage в простому вигляді — рейтинг рахується по
    всій історії включно з майбутнім. Для курсової згадуємо як
    спрощення; для production правильно перерахувати рейтинг для
    кожного тендера лише на основі попередніх.
    """
    print("\n[Рейтинг] Рахуємо buyer_violation_rate...")
    stats = df.groupby("buyer_id").agg(
        n_tenders   =("tender_id",  "count"),
        n_violations=("dasu_label", "sum"),
    ).reset_index()

    ALPHA, BETA = 1, 10
    stats["buyer_violation_rate"] = (
        (stats["n_violations"] + ALPHA) / (stats["n_tenders"] + BETA)
    ).astype("float32")

    print(f"[Рейтинг] Унікальних замовників: {len(stats):,}")
    print(f"[Рейтинг] Розподіл buyer_violation_rate:")
    print(stats["buyer_violation_rate"].describe())

    return df.merge(
        stats[["buyer_id", "buyer_violation_rate", "n_tenders"]]
            .rename(columns={"n_tenders": "buyer_total_tenders"}),
        on="buyer_id", how="left"
    )

def add_normalized_features(df: pd.DataFrame) -> pd.DataFrame:
    """Нормалізація tender_value і submission_window відносно CPV та buyer."""
    print("\n[Нормалізація] Рахуємо медіани...")

    # 1. По CPV
    cpv_med_value  = df.groupby("main_cpv_2_digit")["log_tender_value"].transform("median")
    df["value_vs_cpv_median"] = (df["log_tender_value"] - cpv_med_value).astype("float32")

    # submission_window — рахуємо лише там де він є
    valid_sw = df["has_submission_window"] == 1
    cpv_med_sw = df.loc[valid_sw].groupby("main_cpv_2_digit")["submission_window_days"].median()
    df["cpv_median_sw"] = df["main_cpv_2_digit"].map(cpv_med_sw).fillna(7).astype("float32")
    df["window_vs_cpv_median"] = (df["submission_window_days"] - df["cpv_median_sw"]).astype("float32")
    df = df.drop(columns=["cpv_median_sw"])

    # 2. По buyer
    buyer_med_value = df.groupby("buyer_id")["log_tender_value"].transform("median")
    df["value_vs_buyer_median"] = (df["log_tender_value"] - buyer_med_value).astype("float32")

    return df


# ══════════════════════════════════════════════════════════════
# 4. Головний цикл
# ══════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import time
    t0 = time.time()

    violation_ids = load_dasu_labels()

    tender_files = sorted(glob.glob(TENDERS_GLOB))
    if not tender_files:
        raise FileNotFoundError(f"Не знайдено файлів: {TENDERS_GLOB}")
    print(f"\n[Тендери] Знайдено файлів: {len(tender_files)}")

    chunks_out = []
    total_rows = 0
    total_pos  = 0

    for file in tqdm(tender_files, desc="Файли"):
        available = pd.read_csv(file, nrows=0).columns.tolist()
        use_cols  = [c for c in NEEDED_COLS if c in available]

        for chunk in pd.read_csv(
            file,
            chunksize=CHUNKSIZE,
            usecols=use_cols,
            low_memory=False,
            dtype={
                "is_buyer_masked":  "Int8",
                "is_weekend":       "Int8",
                "is_q4":            "Int8",
                "is_december":      "Int8",
                "main_cpv_2_digit": "Int16",
            },
        ):
            out = process_chunk(chunk, violation_ids)
            if not out.empty:
                chunks_out.append(out)
                total_rows += len(out)
                total_pos  += int(out["dasu_label"].sum())

    print(f"\n[Збираємо] Об'єднуємо {len(chunks_out)} чанків...")
    df_tenders = pd.concat(chunks_out, ignore_index=True)
    del chunks_out

    df_tenders = add_buyer_violation_rate(df_tenders)
    df_tenders = add_normalized_features(df_tenders)

    elapsed = time.time() - t0
    pos = int(df_tenders["dasu_label"].sum())
    neg = len(df_tenders) - pos

    print(f"\n{'═' * 55}")
    print(f"  Тендерів усього:        {len(df_tenders):,}")
    print(f"  З порушенням (label=1): {pos:,} ({pos/len(df_tenders):.3%})")
    print(f"  Без порушення (label=0):{neg:,}")
    print(f"  Унікальних замовників:  {df_tenders['buyer_id'].nunique():,}")
    print(f"  Часовий діапазон:       {df_tenders['tender_start_date'].min().date()} → "
          f"{df_tenders['tender_start_date'].max().date()}")
    print(f"  Розмір у пам'яті:       "
          f"{df_tenders.memory_usage(deep=True).sum() / 1024**2:.0f} МБ")
    print(f"  Час:                    {elapsed:.0f}с ({elapsed/60:.1f} хв)")
    print(f"{'═' * 55}")

    df_tenders.to_parquet(OUTPUT_PATH, compression="snappy", engine="pyarrow")
    print(f"\n✅ Збережено: {OUTPUT_PATH}")

[ДАСУ] Завантажено 78,132 тендерів з моніторингом
[ДАСУ] З них з порушеннями: 56,695

[Тендери] Знайдено файлів: 4


Файли: 100%|██████████| 4/4 [01:55<00:00, 28.78s/it]



[Збираємо] Об'єднуємо 45 чанків...

[Рейтинг] Рахуємо buyer_violation_rate...
[Рейтинг] Унікальних замовників: 21,555
[Рейтинг] Розподіл buyer_violation_rate:
count    21555.000000
mean         0.059066
std          0.036077
min          0.000782
25%          0.031250
50%          0.058824
75%          0.083333
max          0.407407
Name: buyer_violation_rate, dtype: float64

[Нормалізація] Рахуємо медіани...

═══════════════════════════════════════════════════════
  Тендерів усього:        922,575
  З порушенням (label=1): 14,428 (1.564%)
  Без порушення (label=0):908,147
  Унікальних замовників:  21,555
  Часовий діапазон:       2021-09-01 → 2025-12-28
  Розмір у пам'яті:       103 МБ
  Час:                    116с (1.9 хв)
═══════════════════════════════════════════════════════

✅ Збережено: features_tenders_full.parquet


In [9]:
df_tenders['dasu_label'].value_counts()
#df_tenders['tender_start_date'].notna().count()

dasu_label
0    908147
1     14428
Name: count, dtype: int64